In [1]:
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import os
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

class DataFetcher:
    def __init__(self, dataset_path: str = "dataset-50.h5", nist_mapping_path: str = "element-map-18a.json"):
        self.dataset_path = dataset_path
        self.nist_mapping_path = nist_mapping_path
        self.load_nist_mapping()
        
    def load_nist_mapping(self):
        """Load NIST element mapping from JSON file"""
        try:
            with open(self.nist_mapping_path, 'r') as f:
                self.nist_mapping = json.load(f)
        except FileNotFoundError:
            print(f"Warning: NIST mapping file {self.nist_mapping_path} not found. Using empty mapping.")
            self.nist_mapping = {}
    
    def fetch_data(self, element: str) -> Optional[pd.DataFrame]:
        """Fetch data for a given element from the HDF5 file"""
        try:
            with h5py.File(self.dataset_path, 'r') as f:
                if element in f:
                    group = f[element]
                    data = {key: group[key][:] for key in group.keys()}
                    return pd.DataFrame(data)
                else:
                    available_elements = list(f.keys())
                    print(f"Element {element} not found. Available elements: {available_elements}")
                    return None
        except Exception as e:
            print(f"Error accessing dataset: {e}")
            return None

class SpectrumSimulator:
    def __init__(self, element: str, data_fetcher: DataFetcher):
        self.element = element
        self.data_fetcher = data_fetcher
        self.data = self.data_fetcher.fetch_data(element)
        self.element_lines = {}
        self.delta_E_max = {}
        
        if self.data is not None:
            self.preprocess_data()
        else:
            print(f"Failed to load data for element {element}")
    
    def preprocess_data(self):
        """Preprocess spectral line data for the element"""
        required_cols = ['Observed', 'Ritz', 'Aki', 'Ei', 'Ek']
        if not all(col in self.data.columns for col in required_cols):
            print(f"Required columns missing for {self.element}")
            return
        
        self.data = self.data.dropna(subset=required_cols)
        self.data = self.data[self.data['Aki'] > 0]
        
        self.data['delta_E'] = abs(self.data['Ek'] - self.data['Ei'])
        self.delta_E_max[self.element] = self.data['delta_E'].max() if len(self.data) > 0 else 0
        
        wavelengths = []
        for _, row in self.data.iterrows():
            if pd.notna(row['Ritz']) and row['Ritz'] > 0:
                wavelengths.append(row['Ritz'])
            elif pd.notna(row['Observed']) and row['Observed'] > 0:
                wavelengths.append(row['Observed'])
        
        if wavelengths:
            self.element_lines[self.element] = {
                'wavelengths': np.array(wavelengths),
                'intensities': self.data['Aki'].values[:len(wavelengths)],
                'lower_energies': self.data['Ei'].values[:len(wavelengths)],
                'upper_energies': self.data['Ek'].values[:len(wavelengths)]
            }

class MixedSpectrumSimulator:
    def __init__(self, elements: List[str], data_fetcher: DataFetcher):
        self.elements = elements
        self.data_fetcher = data_fetcher
        self.simulators = {}
        self.delta_E_max = {}
        
        for element in elements:
            simulator = SpectrumSimulator(element, data_fetcher)
            if simulator.data is not None and element in simulator.element_lines:
                self.simulators[element] = simulator
                self.delta_E_max.update(simulator.delta_E_max)
                
        if not self.simulators:
            print("No valid simulators created.")
    
    def validate_lte(self, temperature: float, selected_elements: List[Tuple[str, float]]) -> Tuple[float, float]:
        delta_E_values = [v for k, v in self.delta_E_max.items() if any(k.startswith(elem) for elem, _ in selected_elements) and v > 0]
        
        if delta_E_values:
            delta_E_max = max(delta_E_values)
            n_e_min = (delta_E_max * 1.60218e-19) / (1.38065e-23 * temperature) * 1e-6
            return delta_E_max, n_e_min
        return 0, 0
    
    def generate_spectrum(self, selected_elements: List[Tuple[str, float]], temperature: float) -> Tuple[np.ndarray, np.ndarray, Dict, List[Tuple[float, float, str]]]:
        """Generate mixed spectrum from selected elements with their percentages"""
        if not self.simulators:
            return np.array([]), np.array([]), {}, []
        
        delta_E_max, n_e_min = self.validate_lte(temperature, selected_elements)
        
        wavelength_range = np.linspace(200, 900, 7001)
        mixed_spectrum = np.zeros_like(wavelength_range)
        all_prominent_lines = []
        atom_percentages = {}
        
        total_percentage = sum(p for _, p in selected_elements)
        if total_percentage == 0:
            return wavelength_range, mixed_spectrum, atom_percentages, all_prominent_lines
        
        for base_elem, percentage in selected_elements:
            if percentage <= 0:
                continue
                
            # Handle multi-ionization states (e.g., "Al" covers "Al I", "Al II", "Al III")
            matching_simulators = {k: v for k, v in self.simulators.items() if k.startswith(base_elem)}
            
            for element, simulator in matching_simulators.items():
                if element not in simulator.element_lines:
                    continue
                
                lines = simulator.element_lines[element]
                wavelengths = lines['wavelengths']
                intensities = lines['intensities']
                
                weight = percentage / total_percentage
                
                # Calculate Boltzmann distribution
                k_B = 8.617333e-5  # eV/K
                lower_energies = lines['lower_energies']
                boltzmann_factors = np.exp(-lower_energies / (k_B * temperature))
                
                # Apply weight and Boltzmann factors
                weighted_intensities = intensities * weight * boltzmann_factors
                
                # Find prominent lines (top 10%)
                intensity_threshold = np.percentile(weighted_intensities, 90)
                prominent_mask = weighted_intensities >= intensity_threshold
                
                for i, (wl, intensity) in enumerate(zip(wavelengths, weighted_intensities)):
                    if 200 <= wl <= 900:
                        # Add line to spectrum with Gaussian broadening
                        sigma = 0.1  # nm
                        gaussian = np.exp(-0.5 * ((wavelength_range - wl) / sigma) ** 2)
                        mixed_spectrum += intensity * gaussian
                        
                        # Store prominent lines for labeling
                        if prominent_mask[i]:
                            all_prominent_lines.append((wl, intensity, f"{element} {wl:.2f}nm"))
                
                # Store atom percentage information
                if base_elem not in atom_percentages:
                    atom_percentages[base_elem] = 0
                atom_percentages[base_elem] += weight * 100
        
        # Sort prominent lines by intensity (descending)
        all_prominent_lines.sort(key=lambda x: x[1], reverse=True)
        
        return wavelength_range, mixed_spectrum, atom_percentages, all_prominent_lines

def plot_spectrum_plotly(wavelengths, spectrum, atom_percentages, all_prominent_lines, temperature, electron_density, show_labels=True):
    """Plot spectrum using Plotly with optional peak labels"""
    fig = go.Figure()
    
    # Add main spectrum trace
    fig.add_trace(go.Scatter(
        x=wavelengths,
        y=spectrum,
        mode='lines',
        name='Mixed Spectrum',
        line=dict(color='blue', width=2)
    ))
    
    # Add peak labels if enabled
    if show_labels and all_prominent_lines:
        # Show only top 15 most prominent lines to avoid overcrowding
        top_lines = all_prominent_lines[:15]
        
        for wl, intensity, label in top_lines:
            # Find the closest point in the spectrum
            idx = np.argmin(np.abs(wavelengths - wl))
            fig.add_annotation(
                x=wl,
                y=spectrum[idx],
                text=label,
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=1,
                arrowcolor="red",
                bgcolor="rgba(255,255,255,0.8)",
                bordercolor="red",
                borderwidth=1,
                font=dict(size=10)
            )
    
    # Update layout
    fig.update_layout(
        title=f'Mixed Atomic Emission Spectrum<br>Temperature: {temperature} K, Electron Density: {electron_density:.1e} cm⁻³',
        xaxis_title='Wavelength (nm)',
        yaxis_title='Intensity (arbitrary units)',
        width=1000,
        height=600,
        showlegend=True
    )
    
    # Add composition info to the plot
    composition_text = "Composition: " + ", ".join([f"{elem}: {perc:.1f}%" for elem, perc in atom_percentages.items()])
    fig.add_annotation(
        text=composition_text,
        xref="paper", yref="paper",
        x=0.02, y=0.98,
        showarrow=False,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1
    )
    
    fig.show()

def create_composition_form():
    """Create interactive form for element composition input"""
    print("\n=== LIBS Spectrum Simulation ===")
    print("Pilih komposisi elemen (dalam %):")
    
    # Available elements
    elements = ['Al', 'Ca', 'Fe', 'Si', 'Mg', 'Ti', 'Cr', 'Mn', 'Ni', 'Cu', 'Li','Ca', 'Mg', 'Na', 'K', 'O', 'H', 'N', 'Sr', 'Si', "C", 'Rb',]
    
    # Create composition widgets
    composition_widgets = []
    for elem in elements:
        label = widgets.Label(value=f'{elem}:')
        slider = widgets.FloatSlider(
            value=0.0,
            min=0.0,
            max=100.0,
            step=0.1,
            description='',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px')
        )
        composition_widgets.append((label, slider))
    
    # Temperature and electron density inputs
    temperature_input = widgets.FloatText(
        value=8000.0,
        description='Temperature (K):',
        style={'description_width': 'initial'}
    )
    
    electron_density_input = widgets.FloatText(
        value=1e17,
        description='Electron Density (cm⁻³):',
        style={'description_width': 'initial'}
    )
    
    # Create checkbox for show labels toggle
    show_labels_checkbox = widgets.Checkbox(
        value=True,
        description='Show Peak Labels',
        style={'description_width': 'initial'}
    )
    
    # Submit button
    submit_button = widgets.Button(
        description='Generate Spectrum',
        button_style='success',
        layout=widgets.Layout(width='200px', height='40px')
    )
    
    # Output widget for results
    output = widgets.Output()
    
    def on_submit(b):
        with output:
            clear_output()
            # Get selected elements
            selected_elements = [(w[0].value, w[1].value) for w in composition_widgets if w[1].value > 0]
            if not selected_elements:
                print("Error: No elements with percentage > 0 selected")
                return
            
            total_percentage = sum(p for _, p in selected_elements)
            if total_percentage > 100:
                print(f"Error: Total percentage ({total_percentage:.1f}%) exceeds 100%")
                return
            
            run_simulation(selected_elements, temperature_input.value, electron_density_input.value, show_labels_checkbox.value)
    
    submit_button.on_click(on_submit)
    
    # Create layout
    composition_box = widgets.VBox([
        widgets.HBox([w[0], w[1]]) for w in composition_widgets
    ])
    
    controls_box = widgets.VBox([
        temperature_input,
        electron_density_input,
        show_labels_checkbox,
        submit_button
    ])
    
    main_box = widgets.HBox([composition_box, controls_box])
    
    # Display all widgets
    display(main_box, output)

def run_simulation(selected_elements, temperature, electron_density, show_labels=True):
    """Run the spectrum simulation"""
    print(f"\nMemulai simulasi dengan parameter yang telah ditentukan:")
    print(f"Komposisi: {selected_elements}")
    print(f"Suhu: {temperature} K")
    print(f"Densitas Elektron: {electron_density:.1e} cm^-3\n")
    
    # Initialize data fetcher
    data_fetcher = DataFetcher()
    
    # Get unique base elements (e.g., from [('Al', 25), ('Fe', 50)] get ['Al', 'Fe'])
    unique_elements = {elem for elem, _ in selected_elements}
    
    # Create simulators for all ionization states of the selected elements
    all_element_variants = []
    for base_elem in unique_elements:
        for ionization in ['I', 'II', 'III']:
            variant = f"{base_elem} {ionization}"
            all_element_variants.append(variant)
    
    mixed_simulator = MixedSpectrumSimulator(all_element_variants, data_fetcher)
    
    if not mixed_simulator.simulators:
        print("No valid simulators created based on the selected elements. Exiting.")
        return
    
    print(f"Successfully loaded simulators for: {list(mixed_simulator.simulators.keys())}")
    
    wavelengths, spectrum, atom_percentages, all_prominent_lines = mixed_simulator.generate_spectrum(selected_elements, temperature)
    
    if len(spectrum) == 0:
        print("No spectrum data generated.")
        return
    
    print(f"Generated spectrum with {len(all_prominent_lines)} prominent lines")
    print("Plotting spectrum...")
    
    # Plot the spectrum with the show_labels parameter
    plot_spectrum_plotly(wavelengths, spectrum, atom_percentages, all_prominent_lines, temperature, electron_density, show_labels)
    
    print("\nSimulation completed!")

def main():
    interactive_mode = True  # Set to False for command line usage
    
    if interactive_mode:
        print("Mode Interaktif: Menggunakan widget untuk input")
        create_composition_form()
    else:
        # Non-interactive mode dengan nilai default
        selected = [("Si", 25.0), ("Al", 25.0), ("Fe", 50.0)]
        temp = 8000  # K
        ne = 1e17  # cm^-3
        
        print(f"\nMemulai simulasi dengan parameter yang telah ditentukan:")
        print(f"Komposisi: {selected}")
        print(f"Suhu: {temp} K")
        print(f"Densitas Elektron: {ne:.1e} cm^-3\n")
        
        run_simulation(selected, temp, ne, show_labels=False)

print("Semua fungsi telah didefinisikan. Jalankan main() untuk memulai aplikasi interaktif.")

Semua fungsi telah didefinisikan. Jalankan main() untuk memulai aplikasi interaktif.


In [2]:
# Test checkbox widget
import ipywidgets as widgets
from IPython.display import display

# Create and display the checkbox
show_labels_checkbox = widgets.Checkbox(
    value=True,
    description='Show Peak Labels',
    disabled=False,
    indent=False
)

print("Creating checkbox widget...")
display(show_labels_checkbox)
print("Checkbox should appear above this message.")

Creating checkbox widget...


Checkbox(value=True, description='Show Peak Labels', indent=False)

Checkbox should appear above this message.


In [3]:
# Jalankan aplikasi interaktif dengan fitur toggle checkbox
main()

Mode Interaktif: Menggunakan widget untuk input

=== LIBS Spectrum Simulation ===
Pilih komposisi elemen (dalam %):


Output()

In [1]:
import numpy as np
import pandas as pd
import json
import torch
import torch.nn.functional as F
from scipy.signal.windows import gaussian
from scipy.signal import find_peaks
import h5py
import re
import plotly.graph_objects as go
from typing import List, Dict, Tuple, Optional
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# Konfigurasi parameter simulasi
SIMULATION_CONFIG = {
    "resolution": 24480,
    "wl_range": (200, 900),
    "sigma": 0.1,
    "target_max_intensity": 0.8,
    "convolution_sigma": 0.1,
}

# Konstanta fisika
PHYSICAL_CONSTANTS = {
    "k_B": 8.617333262145e-5,  # eV/K
    "m_e": 9.1093837e-31,      # kg
    "h": 4.135667696e-15,      # eV·s
}

# Elemen target
BASE_ELEMENTS = ['Al', 'Ca', 'Fe', 'Si', 'Mg', 'Ti', 'Cr', 'Mn', 'Ni', 'Cu', 'Li','Ca', 'Mg', 'Na', 'K', 'O', 'H', 'N', 'Sr', 'Si', "C", 'Rb',"Co", "Pb"]

def calculate_lte_electron_density(temp: float, delta_E: float) -> float:
    return 1.6e12 * (temp ** 0.5) * (delta_E ** 3)

class DataFetcher:
    def __init__(self, hdf_path: str):
        self.hdf_path = hdf_path
        self.delta_E_max = {}

    def get_nist_data(self, element: str, sp_num: int) -> Tuple[List[List], float]:
        try:
            with pd.HDFStore(self.hdf_path, mode='r') as store:
                df = store.get('nist_spectroscopy_data')
                filtered_df = df[(df['element'] == element) & (df['sp_num'] == sp_num)]
                required_columns = ['ritz_wl_air(nm)', 'Aki(s^-1)', 'Ek(eV)', 'Ei(eV)', 'g_i', 'g_k']
                if filtered_df.empty or not all(col in df.columns for col in required_columns):
                    return [], 0.0
                filtered_df = filtered_df.dropna(subset=required_columns)

                filtered_df['ritz_wl_air(nm)'] = pd.to_numeric(filtered_df['ritz_wl_air(nm)'], errors='coerce')
                for col in ['Ek(eV)', 'Ei(eV)']:
                    filtered_df[col] = filtered_df[col].apply(
                        lambda x: float(re.sub(r'[^\d.-]', '', str(x))) if re.sub(r'[^\d.-]', '', str(x)) else None
                    )
                filtered_df = filtered_df.dropna(subset=['ritz_wl_air(nm)', 'Ek(eV)', 'Ei(eV)'])

                filtered_df = filtered_df[
                    (filtered_df['ritz_wl_air(nm)'] >= SIMULATION_CONFIG["wl_range"][0]) &
                    (filtered_df['ritz_wl_air(nm)'] <= SIMULATION_CONFIG["wl_range"][1])
                ]
                filtered_df['delta_E'] = abs(filtered_df['Ek(eV)'] - filtered_df['Ei(eV)'])
                if not filtered_df.empty:
                    filtered_df = filtered_df.sort_values(by='Aki(s^-1)', ascending=False)
                    delta_E_max = filtered_df['delta_E'].max()
                    if pd.isna(delta_E_max): delta_E_max = 0.0
                else:
                    delta_E_max = 0.0
                self.delta_E_max[f"{element}_{sp_num}"] = delta_E_max
                return filtered_df[required_columns + ['Acc']].values.tolist(), delta_E_max
        except Exception as e:
            print(f"Error fetching NIST data for {element}_{sp_num}: {str(e)}")
            return [], 0.0

class SpectrumSimulator:
    def __init__(
        self,
        nist_data: List[List],
        element: str,
        ion: int,
        temperature: float,
        ionization_energy: float,
        config: Dict = SIMULATION_CONFIG
    ):
        self.nist_data = nist_data
        self.element = element
        self.ion = ion
        self.temperature = temperature
        self.ionization_energy = ionization_energy
        self.resolution = config["resolution"]
        self.wl_range = config["wl_range"]
        self.sigma = config["sigma"]
        self.wavelengths = np.linspace(self.wl_range[0], self.wl_range[1], self.resolution, dtype=np.float32)
        self.gaussian_cache = {}
        self.element_label = f"{element} {'I' if self.ion == 1 else 'II'}"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def partition_function(self, energy_levels: List[float], degeneracies: List[float]) -> float:
        k_B = PHYSICAL_CONSTANTS["k_B"]
        return sum(g * np.exp(-E / (k_B * self.temperature)) for g, E in zip(degeneracies, energy_levels) if E is not None) or 1.0

    def calculate_intensity(self, energy: float, degeneracy: float, einstein_coeff: float, Z: float) -> float:
        k_B = PHYSICAL_CONSTANTS["k_B"]
        return (degeneracy * einstein_coeff * np.exp(-energy / (k_B * self.temperature))) / Z

    def gaussian_profile(self, center: float) -> np.ndarray:
        if center not in self.gaussian_cache:
            x_tensor = torch.tensor(self.wavelengths, device=self.device, dtype=torch.float32)
            center_tensor = torch.tensor(center, device=self.device, dtype=torch.float32)
            sigma_tensor = torch.tensor(self.sigma, device=self.device, dtype=torch.float32)
            gaussian_val = torch.exp(-0.5 * ((x_tensor - center_tensor) / sigma_tensor) ** 2) / (sigma_tensor * torch.sqrt(torch.tensor(2 * np.pi, device=self.device)))
            self.gaussian_cache[center] = gaussian_val.cpu().numpy().astype(np.float32)
        return self.gaussian_cache[center]

    def simulate(self, atom_percentage: float = 1.0) -> Tuple[np.ndarray, np.ndarray, List[Tuple[float, float, str]]]:
        spectrum = torch.zeros(self.resolution, device=self.device, dtype=torch.float32)
        levels = {}
        prominent_lines = []

        for data in self.nist_data:
            try:
                _, _, Ek, Ei, gi, gk, _ = data
                if all(v is not None for v in [Ek, Ei, gi, gk]):
                    levels.setdefault(float(Ei), float(gi))
                    levels.setdefault(float(Ek), float(gk))
            except (ValueError, TypeError):
                continue

        if not levels: return self.wavelengths, np.zeros(self.resolution, dtype=np.float32), []

        energy_levels, degeneracies = list(levels.keys()), list(levels.values())
        Z = self.partition_function(energy_levels, degeneracies)

        for data in self.nist_data:
            try:
                wl, Aki, Ek, _, _, gk, _ = data
                if all(v is not None for v in [wl, Aki, Ek, gk]):
                    wl, Aki, Ek, gk = float(wl), float(Aki), float(Ek), float(gk)
                    intensity = self.calculate_intensity(Ek, gk, Aki, Z)
                    prominent_lines.append((wl, intensity * atom_percentage, self.element_label))

                    idx = np.searchsorted(self.wavelengths, wl)
                    if 0 <= idx < self.resolution:
                        gaussian_contrib = torch.tensor(
                            intensity * atom_percentage * self.gaussian_profile(wl),
                            device=self.device, dtype=torch.float32
                        )
                        start_idx = max(0, idx - len(gaussian_contrib) // 2)
                        end_idx = min(self.resolution, start_idx + len(gaussian_contrib))
                        if start_idx < end_idx:
                            spectrum[start_idx:end_idx] += gaussian_contrib[:end_idx - start_idx]
            except (ValueError, TypeError):
                continue

        prominent_lines.sort(key=lambda x: x[1], reverse=True)
        return self.wavelengths, spectrum.cpu().numpy(), prominent_lines[:20]

class MixedSpectrumSimulator:
    def __init__(self, simulators: List[SpectrumSimulator], electron_density: float, delta_E_max: Dict[str, float], config: Dict = SIMULATION_CONFIG):
        self.simulators = simulators
        self.resolution = config["resolution"]
        self.wl_range = config["wl_range"]
        self.convolution_sigma = config["convolution_sigma"]
        self.electron_density = electron_density
        self.delta_E_max = delta_E_max
        self.wavelengths = np.linspace(self.wl_range[0], self.wl_range[1], self.resolution, dtype=np.float32)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def normalize_intensity(self, intensity: np.ndarray, target_max: float) -> np.ndarray:
        intensity_tensor = torch.tensor(intensity, device=self.device, dtype=torch.float32)
        max_intensity = torch.max(torch.abs(intensity_tensor))
        return intensity if max_intensity == 0 else (intensity_tensor / max_intensity * target_max).cpu().numpy()

    def convolve_spectrum(self, spectrum: np.ndarray, sigma_nm: float) -> np.ndarray:
        spectrum_tensor = torch.tensor(spectrum, device=self.device, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        wavelength_step = (self.wavelengths[-1] - self.wavelengths[0]) / (len(self.wavelengths) - 1)
        sigma_points = sigma_nm / wavelength_step
        kernel_size = int(6 * sigma_points) | 1
        kernel_np = gaussian(kernel_size, sigma_points)
        kernel = torch.tensor(kernel_np / kernel_np.sum(), device=self.device, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        convolved = F.conv1d(spectrum_tensor, kernel, padding=kernel_size//2).squeeze().cpu().numpy()
        return convolved.astype(np.float32)

    def saha_ratio(self, ion_energy: float, temp: float) -> float:
        k_B_eV = PHYSICAL_CONSTANTS["k_B"]
        m_e_kg = PHYSICAL_CONSTANTS["m_e"]
        h_eVs = PHYSICAL_CONSTANTS["h"]
        eV_to_J = 1.60217662e-19
        k_B_J = k_B_eV * eV_to_J
        h_J = h_eVs * eV_to_J
        kT_J = k_B_J * temp
        thermal_term = (2 * np.pi * m_e_kg * kT_J / (h_J**2)) ** 1.5
        n_e_m3 = self.electron_density * 1e6
        pre_exponential_factor = (2.0 / 1.0) * thermal_term / n_e_m3
        exp_term = np.exp(-ion_energy / (k_B_eV * temp))
        return pre_exponential_factor * exp_term

    def validate_lte(self, temperature: float, selected_elements: List[Tuple[str, float]]) -> Tuple[float, float]:
        delta_E_values = [v for k, v in self.delta_E_max.items() if any(k.startswith(elem) for elem, _ in selected_elements) and v > 0]
        delta_E_max = max(delta_E_values) if delta_E_values else 4.0
        n_e_min = calculate_lte_electron_density(temperature, delta_E_max)
        return delta_E_max, n_e_min

    def generate_spectrum(self, selected_elements: List[Tuple[str, float]], temperature: float) -> Tuple[np.ndarray, np.ndarray, Dict, List[Tuple[float, float, str]]]:
        mixed_spectrum = np.zeros(self.resolution, dtype=np.float32)
        all_prominent_lines = []
        atom_percentages_dict = {}

        delta_E_max, n_e_min = self.validate_lte(temperature, selected_elements)
        if self.electron_density < n_e_min:
            print(f"Warning: Electron density {self.electron_density:.1e} cm^-3 is below LTE requirement "
                  f"({n_e_min:.1e} cm^-3) for T={temperature:.0f} K, ΔE_max={delta_E_max:.2f} eV")
            self.electron_density = n_e_min
            print(f"Using n_e = {self.electron_density:.1e} cm^-3 to satisfy LTE")

        total_percentage = sum(p for _, p in selected_elements)
        if abs(total_percentage - 100.0) > 1e-6:
            raise ValueError(f"Total percentage ({total_percentage:.1f}%) must be exactly 100%")

        for base_elem, percentage in selected_elements:
            ion_energy = ionization_energies.get(f"{base_elem} I", 0.0)
            if ion_energy == 0.0:
                print(f"Warning: No ionization energy for {base_elem} I, skipping.")
                atom_percentages_dict[f"{base_elem}_1"] = 0
                atom_percentages_dict[f"{base_elem}_2"] = 0
                continue
            
            saha_val = self.saha_ratio(ion_energy, temperature)
            neutral_fraction = 1 / (1 + saha_val)
            ion_fraction = 1 - neutral_fraction
            atom_percentages_dict[f"{base_elem}_1"] = percentage * neutral_fraction / 100.0
            atom_percentages_dict[f"{base_elem}_2"] = percentage * ion_fraction / 100.0

        for simulator in self.simulators:
            elem_label_key = f"{simulator.element}_{simulator.ion}"
            if elem_label_key in atom_percentages_dict:
                simulator.temperature = temperature
                _, spectrum, prominent_lines = simulator.simulate(atom_percentages_dict[elem_label_key])
                mixed_spectrum += spectrum
                all_prominent_lines.extend(prominent_lines)

        if np.max(mixed_spectrum) == 0:
            print(f"Warning: No spectrum generated for temperature {temperature}")
            return self.wavelengths, np.zeros(self.resolution, dtype=np.float32), {}, []

        convolved_spectrum = self.convolve_spectrum(mixed_spectrum, self.convolution_sigma)
        normalized_spectrum = self.normalize_intensity(convolved_spectrum, SIMULATION_CONFIG["target_max_intensity"])

        final_composition = {k.replace('_', ' '): v * 100 for k, v in atom_percentages_dict.items()}
        final_composition['temperature'] = temperature
        final_composition['electron_density'] = self.electron_density
        final_composition['delta_E_max'] = delta_E_max
        final_composition['n_e_min'] = n_e_min

        return self.wavelengths, normalized_spectrum, final_composition, all_prominent_lines

def plot_spectrum_plotly(wavelengths, spectrum, temperature, electron_density, atom_percentages, all_prominent_lines, show_labels=True):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=wavelengths, y=spectrum, mode='lines', name='Simulated Spectrum', line=dict(color='navy', width=1.5)))

    if np.max(spectrum) > 0 and show_labels:
      # Anda bisa mengatur 'height' untuk mengontrol seberapa banyak puncak yang akan diberi label.
      # Nilai yang lebih kecil akan menampilkan lebih banyak label.
      height_threshold = 0.001 * np.max(spectrum)
      peak_indices, _ = find_peaks(spectrum, height=height_threshold)
      peak_wavelengths = wavelengths[peak_indices]
      peak_intensities = spectrum[peak_indices]

      annotations = []
      all_prominent_lines.sort(key=lambda x: x[1], reverse=True)

      for wl, intensity in zip(peak_wavelengths, peak_intensities):
          if all_prominent_lines:
              closest_line = min(all_prominent_lines, key=lambda line: abs(line[0] - wl))
              label = f"{closest_line[2]}<br>{closest_line[0]:.2f} nm"
              annotations.append(go.layout.Annotation(
                  x=wl, y=intensity, text=label, showarrow=True, arrowhead=2, arrowsize=1,
                  arrowwidth=1, ax=0, ay=-40, font=dict(size=9, color="#ffffff"),
                  align="center", bordercolor="#555", borderwidth=1, borderpad=2,
                  bgcolor="#ff7f0e", opacity=0.8))
      fig.update_layout(annotations=annotations)

    comp_items = [f'<b>{elem}</b>: {perc:.2f}%' for elem, perc in atom_percentages.items() if elem not in ['temperature', 'electron_density', 'delta_E_max', 'n_e_min']]
    param_items = [
        f'<b>&Delta;E_max</b>: {atom_percentages.get("delta_E_max", 0):.2f} eV',
        f'<b>n_e_min</b>: {atom_percentages.get("n_e_min", 0):.1e} cm<sup>-3</sup>']
    comp_text = '<b>Composition:</b><br>' + '<br>'.join(comp_items) + '<br><br>' + '<br>'.join(param_items)

    fig.update_layout(
        title=f'<b>Simulated Spectrum</b><br>T = {temperature:.0f} K, n_e = {electron_density:.1e} cm<sup>-3</sup>',
        xaxis_title='Wavelength (nm)', yaxis_title='Normalized Intensity (a.u.)',
        xaxis=dict(range=[wavelengths[0], wavelengths[-1]], showgrid=True, gridwidth=1, gridcolor='LightGray', minor=dict(showgrid=True, gridcolor='lightgrey', griddash='dot')),
        yaxis=dict(range=[-0.05, max(spectrum.max() * 1.15, 0.1)], showgrid=True, gridwidth=1, gridcolor='LightGray', minor=dict(showgrid=True, gridcolor='lightgrey', griddash='dot')),
        legend=dict(x=0.01, y=0.99, bordercolor='Black', borderwidth=1),
        font=dict(family="Arial, sans-serif", size=12, color="black"),
        plot_bgcolor='white', paper_bgcolor='white',
        annotations=fig.layout.annotations + (go.layout.Annotation(
            x=0.98, y=0.98, xref='paper', yref='paper', text=comp_text, showarrow=False,
            align='left', valign='top', font=dict(size=10), bgcolor='rgba(255, 255, 255, 0.8)',
            bordercolor='black', borderwidth=1),)
    )
    fig.show()

def create_composition_form():
    element_options = [(elem, elem) for elem in BASE_ELEMENTS]
    composition_widgets = []
    total_percentage_label = widgets.Label(value="Total Percentage: 0.0%")

    def add_element_row(_=None):
        element_dropdown = widgets.Dropdown(options=element_options, description="Element:", layout={'width': '300px'})
        percentage_input = widgets.FloatText(value=0.0, description="Percentage (%):", layout={'width': '200px'})
        remove_button = widgets.Button(description="Remove", button_style='danger')

        def update_total(_=None):
            total = sum(w[1].value for w in composition_widgets)
            total_percentage_label.value = f"Total Percentage: {total:.1f}%"

        def remove_row(b):
            row_to_remove = next((row for row in composition_widgets if row[2] == b), None)
            if row_to_remove:
                composition_widgets.remove(row_to_remove)
                rows_vbox.children = [widgets.HBox(list(row)) for row in composition_widgets]
                update_total()

        percentage_input.observe(update_total, names='value')
        remove_button.on_click(remove_row)
        composition_widgets.append((element_dropdown, percentage_input, remove_button))
        rows_vbox.children = [widgets.HBox(list(row)) for row in composition_widgets]
        update_total()

    add_button = widgets.Button(description="Add Element", button_style='success')
    add_button.on_click(add_element_row)

    rows_vbox = widgets.VBox([])
    control_hbox = widgets.HBox([add_button, total_percentage_label])
    display(widgets.VBox([rows_vbox, control_hbox]))
    add_element_row()

    submit_button = widgets.Button(description="Generate Spectrum", button_style='primary')
    
    # Create checkbox for show labels toggle
    show_labels_checkbox = widgets.Checkbox(
        value=True,
        description='Show Peak Labels',
        style={'description_width': 'initial'}
    )
    
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output()
            selected_elements = [(w[0].value, w[1].value) for w in composition_widgets if w[1].value > 0]
            if not selected_elements:
                print("Error: No elements with percentage > 0 selected")
                return
            
            total_percentage = sum(p for _, p in selected_elements)
            if abs(total_percentage - 100.0) > 1e-6:
                print(f"Error: Total percentage ({total_percentage:.1f}%) must be exactly 100%")
                return

            run_simulation(selected_elements, temperature_input.value, electron_density_input.value, show_labels_checkbox.value)

    submit_button.on_click(on_submit)
    temperature_input = widgets.FloatText(value=10000, description="Temperature (K):", layout={'width': '300px'})

    ne_options_values = [10**exp for exp in np.arange(15, 18.1, 0.1)]
    ne_options_labels = [f"{val:.1e}" for val in ne_options_values]
    ne_options = list(zip(ne_options_labels, ne_options_values))
    desired_default = 1e17
    actual_default = min(ne_options_values, key=lambda x: abs(x - desired_default))
    electron_density_input = widgets.Dropdown(options=ne_options, value=actual_default, description="Electron Density (cm^-3):", layout={'width': '400px'})

    display(temperature_input, electron_density_input, show_labels_checkbox, submit_button, output)

def run_simulation(selected_elements, temperature, electron_density, show_labels=True):
    data_dir = "data"
    # Perbaikan nama file agar konsisten
    nist_path = "data/nist_data(1).h5"
    atomic_data_path = "data/atomic_data1.h5"

    if not os.path.exists(nist_path) or not os.path.exists(atomic_data_path):
        print(f"Error: Pastikan file '{os.path.basename(nist_path)}' dan '{os.path.basename(atomic_data_path)}' ada di dalam folder '{data_dir}/'")
        return

    global ionization_energies
    ionization_energies = {}
    with h5py.File(atomic_data_path, 'r') as f:
        dset = f['elements']
        columns = dset.attrs['columns']
        data = []
        for item in dset[:]:
            row = [
                item[0],
                item[1].decode('utf-8'), item[2].decode('utf-8'),
                item[3].decode('utf-8'), item[4].decode('utf-8'),
                item[5], item[6].decode('utf-8')
            ]
            data.append(row)
        df_ionization = pd.DataFrame(data, columns=columns)
        for _, row in df_ionization.iterrows():
            ionization_energies[row["Sp. Name"]] = float(row["Ionization Energy (eV)"])

    fetcher = DataFetcher(nist_path)
    simulators = []
    delta_E_max_dict = {}
    unique_elements = {elem for elem, _ in selected_elements}

    for elem in unique_elements:
        for ion in [1, 2]:
            nist_data, delta_E = fetcher.get_nist_data(elem, ion)
            if not nist_data:
                continue
            delta_E_max_dict[f"{elem}_{ion}"] = delta_E
            ion_energy = ionization_energies.get(f"{elem} {'I' if ion == 1 else 'II'}", 0.0)
            simulator = SpectrumSimulator(nist_data, elem, ion, temperature, ion_energy)
            simulators.append(simulator)

    if not simulators:
        print("No valid simulators created based on the selected elements. Exiting.")
        return

    mixed_simulator = MixedSpectrumSimulator(simulators, electron_density, delta_E_max_dict)
    wavelengths, spectrum, atom_percentages, all_prominent_lines = mixed_simulator.generate_spectrum(selected_elements, temperature)

    if atom_percentages:
        print(f"Spectrum generated with:")
        print(f"Temperature: {temperature:.0f} K")
        print(f"Electron Density: {atom_percentages['electron_density']:.1e} cm^-3")
        print(f"ΔE_max: {atom_percentages['delta_E_max']:.2f} eV")
        print(f"n_e_min: {atom_percentages['n_e_min']:.1e} cm^-3")
        print("Atomic Composition (after Saha adjustment):")
        for elem, percentage in atom_percentages.items():
            if elem not in ['temperature', 'electron_density', 'delta_E_max', 'n_e_min']:
                print(f"  {elem}: {percentage:.2f}%")

        plot_spectrum_plotly(wavelengths, spectrum, temperature, atom_percentages['electron_density'], atom_percentages, all_prominent_lines, show_labels)

def main():
    if 'ipykernel' in __import__('sys').modules:
        print("Menjalankan dalam mode interaktif (Jupyter/IPython). Menampilkan UI.")
        create_composition_form()
    else:
        print("Menjalankan dalam mode skrip non-interaktif.")
        selected = [("Si", 25.0), ("Al", 25.0), ("Fe", 50.0)]
        temp = 12000
        ne = 1e17
        print(f"\nMemulai simulasi dengan parameter yang telah ditentukan:")
        print(f"Komposisi: {selected}")
        print(f"Suhu: {temp} K")
        print(f"Densitas Elektron: {ne:.1e} cm^-3\n")
        run_simulation(selected, temp, ne, show_labels=False)

if __name__ == "__main__":
    main()

Menjalankan dalam mode interaktif (Jupyter/IPython). Menampilkan UI.


FloatText(value=10000.0, description='Temperature (K):', layout=Layout(width='300px'))

Dropdown(description='Electron Density (cm^-3):', index=20, layout=Layout(width='400px'), options=(('1.0e+15',…

Checkbox(value=True, description='Show Peak Labels', style=CheckboxStyle(description_width='initial'))

Button(button_style='primary', description='Generate Spectrum', style=ButtonStyle())

Output()

## 4-Panel Visualization: LIBS Analysis Challenges (Matplotlib)


This section creates a 2x2 figure using `simulate_spectrum(elements_dict, temperature, electron_density)` which returns `(wavelengths, intensities)`.



Panels:

1) Pure spectra of three elements (Fe, Ti, Ca) at T=8000 K, n_e=1e17 — each normalized, legend shows element, temperature, and concentration

2) Combined spectrum (Fe+Ti+Ca) at T=8000 K — normalized, legend includes concentrations

3) Zoomed view (420–450 nm) showing the three pure elements (Fe, Ti, Ca) overlaid (scaled to Fe=0.34, Ti=0.33, Ca=0.33) plus the mixture (black)

4) Temperature variability: overlay of the same Fe+Ti+Ca composition at 6000 K, 9000 K, and 12000 K — each spectrum normalized; x-axis limited to 395–440 nm



Note: X-axis limits — panel 1 and 2: 220–550 nm; panel 3: 420–450 nm; panel 4: 395–440 nm.

In [2]:
# 4-Panel LIBS visualization (Matplotlib) — uses existing reference API via a thin wrapper

import numpy as np
import matplotlib.pyplot as plt
import os
import h5py
import pandas as pd


# ----- Helper: normalize safely -----
def _normalize(y):
    y = np.asarray(y)
    m = np.nanmax(np.abs(y)) if y.size else 0.0
    return (y / m) if m > 0 else y


# ----- Thin wrapper to the reference implementation -----
# This matches the requested contract: simulate_spectrum(elements_dict, temperature, electron_density)
def simulate_spectrum(elements_dict: dict, temperature: float, electron_density: float):
    # Ensure reference classes are available (from the acuan cell)
    try:
        DataFetcher  # type: ignore
        SpectrumSimulator  # type: ignore
        MixedSpectrumSimulator  # type: ignore
    except NameError:
        raise RuntimeError("Reference not executed. Run the cell defining DataFetcher, SpectrumSimulator, and MixedSpectrumSimulator.")

    # Resolve data paths as used by the reference cell
    data_dir = "data"
    nist_path = os.path.join(data_dir, "nist_data(1).h5")
    atomic_data_path = os.path.join(data_dir, "atomic_data1.h5")
    if not (os.path.exists(nist_path) and os.path.exists(atomic_data_path)):
        raise FileNotFoundError(f"Missing data files. Expected '{nist_path}' and '{atomic_data_path}'.")

    # Build ionization energy map expected by MixedSpectrumSimulator.generate_spectrum()
    global ionization_energies
    ionization_energies = {}
    with h5py.File(atomic_data_path, 'r') as f:
        dset = f['elements']
        columns = dset.attrs['columns']
        data_rows = []
        for item in dset[:]:
            row = [
                item[0],
                item[1].decode('utf-8'), item[2].decode('utf-8'),
                item[3].decode('utf-8'), item[4].decode('utf-8'),
                item[5], item[6].decode('utf-8')
            ]
            data_rows.append(row)
        df_ion = pd.DataFrame(data_rows, columns=columns)
        for _, r in df_ion.iterrows():
            ionization_energies[r["Sp. Name"]] = float(r["Ionization Energy (eV)"])

    # Prepare selected elements as percentages summing to 100
    elems = list(elements_dict.keys())
    vals = np.array(list(elements_dict.values()), dtype=float)
    if vals.sum() <= 0:
        raise ValueError("elements_dict must contain positive weights")
    percentages = (vals / vals.sum()) * 100.0
    selected_elements = list(zip(elems, percentages))

    # Initialize reference fetcher and simulators
    fetcher = DataFetcher(nist_path)
    simulators = []
    delta_E_max_dict = {}
    unique_elems = {e for e, _ in selected_elements}
    for elem in unique_elems:
        for ion in [1, 2]:
            nist_data, delta_E = fetcher.get_nist_data(elem, ion)
            if not nist_data:
                continue
            delta_E_max_dict[f"{elem}_{ion}"] = delta_E
            ion_energy = ionization_energies.get(f"{elem} {'I' if ion == 1 else 'II'}", 0.0)
            sim = SpectrumSimulator(nist_data, elem, ion, temperature, ion_energy)
            simulators.append(sim)

    if not simulators:
        return np.array([]), np.array([])

    mixer = MixedSpectrumSimulator(simulators, electron_density, delta_E_max_dict)
    wavelengths, spectrum, _, _ = mixer.generate_spectrum(selected_elements, temperature)
    return wavelengths, spectrum


# ----- NEW: compute per-element component spectra that sum to the mixture -----
def simulate_components(elements_dict: dict, temperature: float, electron_density: float):
    """Return (wavelengths, mixture_norm, components_norm)
    components_norm is a dict {base_element: spectrum}, and all components are convolved
    and normalized with the same factor as the mixture so their sum equals the mixture.
    """
    # Ensure reference classes are available
    try:
        DataFetcher  # type: ignore
        SpectrumSimulator  # type: ignore
        MixedSpectrumSimulator  # type: ignore
    except NameError:
        raise RuntimeError("Reference not executed. Run the cell defining DataFetcher, SpectrumSimulator, and MixedSpectrumSimulator.")

    data_dir = "data"
    nist_path = os.path.join(data_dir, "nist_data(1).h5")
    atomic_data_path = os.path.join(data_dir, "atomic_data1.h5")
    if not (os.path.exists(nist_path) and os.path.exists(atomic_data_path)):
        raise FileNotFoundError(f"Missing data files. Expected '{nist_path}' and '{atomic_data_path}'.")

    # Load ionization energies
    global ionization_energies
    ionization_energies = {}
    with h5py.File(atomic_data_path, 'r') as f:
        dset = f['elements']
        columns = dset.attrs['columns']
        data_rows = []
        for item in dset[:]:
            row = [
                item[0],
                item[1].decode('utf-8'), item[2].decode('utf-8'),
                item[3].decode('utf-8'), item[4].decode('utf-8'),
                item[5], item[6].decode('utf-8')
            ]
            data_rows.append(row)
        df_ion = pd.DataFrame(data_rows, columns=columns)
        for _, r in df_ion.iterrows():
            ionization_energies[r["Sp. Name"]] = float(r["Ionization Energy (eV)"])

    # Normalize composition to 100%
    elems = list(elements_dict.keys())
    vals = np.array(list(elements_dict.values()), dtype=float)
    if vals.sum() <= 0:
        raise ValueError("elements_dict must contain positive weights")
    percentages = (vals / vals.sum()) * 100.0
    selected_elements = list(zip(elems, percentages))

    # Build simulators
    fetcher = DataFetcher(nist_path)
    simulators = []
    delta_E_max_dict = {}
    for elem in elems:
        for ion in [1, 2]:
            nist_data, delta_E = fetcher.get_nist_data(elem, ion)
            if not nist_data:
                continue
            delta_E_max_dict[f"{elem}_{ion}"] = delta_E
            ion_energy = ionization_energies.get(f"{elem} {'I' if ion == 1 else 'II'}", 0.0)
            simulators.append(SpectrumSimulator(nist_data, elem, ion, temperature, ion_energy))

    if not simulators:
        return np.array([]), np.array([]), {}

    mixer = MixedSpectrumSimulator(simulators, electron_density, delta_E_max_dict)

    # LTE check and use adjusted electron density if needed (consistent with generate_spectrum)
    delta_E_max, n_e_min = mixer.validate_lte(temperature, selected_elements)
    if mixer.electron_density < n_e_min:
        mixer.electron_density = n_e_min
    # Saha-based neutral/ion fractions per base element
    atom_percentages_dict = {}
    for base_elem, pct in selected_elements:
        ion_energy = ionization_energies.get(f"{base_elem} I", 0.0)
        if ion_energy == 0.0:
            atom_percentages_dict[f"{base_elem}_1"] = 0.0
            atom_percentages_dict[f"{base_elem}_2"] = 0.0
            continue
        saha_val = mixer.saha_ratio(ion_energy, temperature)
        neutral_fraction = 1.0 / (1.0 + saha_val)
        ion_fraction = 1.0 - neutral_fraction
        atom_percentages_dict[f"{base_elem}_1"] = pct * neutral_fraction / 100.0
        atom_percentages_dict[f"{base_elem}_2"] = pct * ion_fraction / 100.0

    # Accumulate spectra per base element and the total
    comp_raw = {e: np.zeros_like(mixer.wavelengths, dtype=np.float32) for e in elems}
    mixed_raw = np.zeros_like(mixer.wavelengths, dtype=np.float32)

    for sim in simulators:
        label_key = f"{sim.element}_{sim.ion}"
        weight = atom_percentages_dict.get(label_key, 0.0)
        if weight <= 0:
            continue
        sim.temperature = temperature
        try:
            _, sp, _ = sim.simulate(weight)  # notebook implementation returns (wl, spectrum, prominent_lines)
        except Exception:
            _, sp = sim.simulate(weight)    # fallback if only (wl, spectrum)
        mixed_raw += sp
        comp_raw[sim.element] += sp

    # Convolve total and components with the same kernel
    mixed_conv = mixer.convolve_spectrum(mixed_raw, mixer.convolution_sigma)
    comp_conv = {e: mixer.convolve_spectrum(s, mixer.convolution_sigma) for e, s in comp_raw.items()}

    # Normalize all by the SAME factor so that sum(components) == mixture and max(mixture)=1
    mmax = float(np.max(np.abs(mixed_conv))) if mixed_conv.size else 0.0
    scale = (1.0 / mmax) if mmax > 0 else 1.0
    mixed_norm = mixed_conv * scale
    comp_norm = {e: s * scale for e, s in comp_conv.items()}

    return mixer.wavelengths, mixed_norm, comp_norm


# ----- Parameters -----
T_ref = 8000
ne_ref = 1e17
x_min, x_max = 220.0, 550.0

# Define composition used throughout (Fe, Ti, Ca)
mix_comp = {'Fe': 0.34, 'Ti': 0.33, 'Ca': 0.33}

# 1) Pure/reference spectra (Fe, Ti, Ca)
wl_fe, I_fe = simulate_spectrum({'Fe': 1.0}, T_ref, ne_ref)
wl_ti, I_ti = simulate_spectrum({'Ti': 1.0}, T_ref, ne_ref)
wl_ca, I_ca = simulate_spectrum({'Ca': 1.0}, T_ref, ne_ref)

I_fe_n = _normalize(I_fe)
I_ti_n = _normalize(I_ti)
I_ca_n = _normalize(I_ca)

# 2) Mixed spectrum at 8000 K
wl_mix_8k, I_mix_8k = simulate_spectrum(mix_comp, T_ref, ne_ref)
I_mix_8k_n = _normalize(I_mix_8k)

# 3) Zoomed region now shows per-element component contributions vs mixture over 420–450 nm
zoom_min, zoom_max = 420.0, 450.0

# 4) Temperature variability (6000/9000/12000 K)
temps = [6000, 9000, 12000]
temp_colors = {6000: 'C4', 9000: 'C6', 12000: 'C3'}
temp_spectra = {}
for T in temps:
    wl_T, I_T = simulate_spectrum(mix_comp, T, ne_ref)
    temp_spectra[T] = (wl_T, _normalize(I_T))

# Also compute true component contributions (convolved + globally normalized)
wl_comp, I_mix_comp_n, comp_norm = simulate_components(mix_comp, T_ref, ne_ref)

# ----- Plotting in 2x2 layout -----
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False)

# Panel 1: Pure spectra — translucent lines, legend includes element, T, and concentration
ax = axes[0,0]
ax.plot(wl_fe, I_fe_n, color='C3', alpha=0.85, lw=1.5, label=f'Fe (T={T_ref} K, conc=100%)')
ax.plot(wl_ti, I_ti_n, color='C1', alpha=0.85, lw=1.5, label=f'Ti (T={T_ref} K, conc=100%)')
ax.plot(wl_ca, I_ca_n, color='C2', alpha=0.85, lw=1.5, label=f'Ca (T={T_ref} K, conc=100%)')
# ax.set_title removed; add letter below instead
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Normalized Intensity (a.u.)')
ax.set_xlim(x_min, x_max)
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
ax.text(0.5, -0.22, 'a', transform=ax.transAxes, ha='center', va='top', fontsize=12, fontweight='bold', clip_on=False)

# Panel 2: Mixed spectrum at 8000 K — legend with composition
ax = axes[0,1]
comp_str = ', '.join([f"{k}={v:.2f}" for k, v in mix_comp.items()])
ax.plot(wl_mix_8k, I_mix_8k_n, color='black', alpha=0.9, lw=1.2, label=f'Mixture ({comp_str})')
# ax.set_title removed; add letter below instead
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Normalized Intensity (a.u.)')
ax.set_xlim(x_min, x_max)
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
ax.text(0.5, -0.22, 'b', transform=ax.transAxes, ha='center', va='top', fontsize=12, fontweight='bold', clip_on=False)

# Panel 3: Overlay true component contributions vs mixture in 420–450 nm
ax = axes[1,0]
mask_fe = (wl_comp >= zoom_min) & (wl_comp <= zoom_max)
mask_ti = mask_fe  # same grid
mask_ca = mask_fe
mask_mix = mask_fe
# Apply +0.015 offset to Fe component only
ax.plot(wl_comp[mask_fe], comp_norm.get('Fe', np.zeros_like(wl_comp))[mask_fe] + 0.015, color='C3', lw=1.3, label='Fe component (+0.015)')
# Label Ti with +0.00 in legend (no offset applied)
ax.plot(wl_comp[mask_ti], comp_norm.get('Ti', np.zeros_like(wl_comp))[mask_ti], color='C1', lw=1.3, label='Ti component (+0.00)')
# Apply +0.03 offset to Ca component only
ax.plot(wl_comp[mask_ca], comp_norm.get('Ca', np.zeros_like(wl_comp))[mask_ca] + 0.03, color='C2', lw=1.3, label='Ca component (+0.03)')
# Apply +0.05 offset to mixture only
ax.plot(wl_comp[mask_mix], I_mix_comp_n[mask_mix] + 0.05, color='black', lw=1.2, alpha=0.9, label='Mixture (+0.05)')
# ax.set_title removed; add letter below instead
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Normalized Intensity (a.u.)')
ax.set_xlim(420.0, 450.0)
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
ax.text(0.5, -0.22, 'c', transform=ax.transAxes, ha='center', va='top', fontsize=12, fontweight='bold', clip_on=False)

# Panel 4: Temperature variability overlay with 395–440 nm window and three temperatures
ax = axes[1,1]
for T in temps:
    wl_T, I_T = temp_spectra[T]
    mask = (wl_T >= 395.0) & (wl_T <= 440.0)
    ax.plot(wl_T[mask], I_T[mask], color=temp_colors[T], alpha=0.9, lw=1.2, label=f'{T} K ({comp_str})')
# ax.set_title removed; add letter below instead
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Normalized Intensity (a.u.)')
ax.set_xlim(395.0, 440.0)
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

# Dynamically find and annotate Ca lines at their actual local peaks using the 9000 K curve
ref_T = 9000 if 9000 in temp_spectra else next(iter(temp_spectra.keys()))
wl_ref, I_ref = temp_spectra[ref_T]

# Helper to annotate nearest local max around target nm with text at fixed height 0.4
def _annotate_peak(label, target_nm, window=1.0, dx=1.2, y_fixed=0.4):
    mask = (wl_ref >= target_nm - window/2) & (wl_ref <= target_nm + window/2)
    if not np.any(mask):
        return
    idx = np.argmax(I_ref[mask])
    x0 = wl_ref[mask][idx]
    y0 = I_ref[mask][idx]
    ax.annotate(label, xy=(x0, y0), xytext=(x0 + dx, y_fixed),
                textcoords='data', arrowprops=dict(arrowstyle='->', color='dimgray', lw=1),
                ha='left', va='center', fontsize=9, color='dimgray')

_annotate_peak('Ca II', 396.0, window=1.0)
_annotate_peak('Ca I', 422.0, window=1.0)

# Add letter below Panel 4
ax.text(0.5, -0.22, 'd', transform=ax.transAxes, ha='center', va='top', fontsize=12, fontweight='bold', clip_on=False)

plt.tight_layout()
# Ensure enough bottom margin so below-text is visible
plt.subplots_adjust(bottom=0.08)

# Export high-DPI PNG for publication
os.makedirs('output', exist_ok=True)
export_path = os.path.join('output', 'libs_4panel.png')
fig.savefig(export_path, dpi=600, bbox_inches='tight', facecolor='white')

plt.show()
print(f"Saved high-DPI PNG to: {export_path}")

RuntimeError: Reference not executed. Run the cell defining DataFetcher, SpectrumSimulator, and MixedSpectrumSimulator.